# Final Project Phase 2 Summary
This Jupyter Notebook (.ipynb) will serve as the skeleton file for your submission for Phase 2 of the Final Project. Answer all statements addressed below as specified in the instructions for the project, covering all necessary details. Please be clear and concise in your answers. Each response should be at most 3 sentences. Good luck! <br><br>

Note: To edit a Markdown cell, double-click on its text.

## Jupyter Notebook Quick Tips
Here are some quick formatting tips to get you started with Jupyter Notebooks. This is by no means exhaustive, and there are plenty of articles to highlight other things that can be done. We recommend using HTML syntax for Markdown but there is also Markdown syntax that is more streamlined and might be preferable.
<a href = "https://towardsdatascience.com/markdown-cells-jupyter-notebook-d3bea8416671">Here's an article</a> that goes into more detail. (Double-click on cell to see syntax)

# Heading 1
## Heading 2
### Heading 3
#### Heading 4
<br>
<b>BoldText</b> or <i>ItalicText</i>
<br> <br>
Math Formulas: $x^2 + y^2 = 1$
<br> <br>
Line Breaks are done using br enclosed in < >.
<br><br>
Hyperlinks are done with: <a> https://www.google.com </a> or
<a href="http://www.google.com">Google</a><br>

# Data Collection and Cleaning
You are required to provide data collection and cleaning for the three (3) minimum datasets. Create a function for each of the following sections that reads or scrapes data from a file or website, manipulate and cleans the parsed data, and writes the cleaned data into a new file.

Make sure your data cleaning and manipulation process is not too simple. Performing complex manipulation and using modules not taught in class shows effort, which will increase the chance of receiving full credit.


## Data Sources
Include sources (as links) to your datasets. Add any additional data sources if needed. Clearly indicate if a data source is different from one submitted in your Phase I, as we will check that it satisfies the requirements.
*   Downloaded Dataset Source:https://collegescorecard.ed.gov/data (download the one titled: “Most Recent Institution- Level Data)
*   Web Collection #1 Source: https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_educational_attainment
*   Web Collection #2 Source: https://api.census.gov/data/2022/acs/acs5/variables.html



## Downloaded Dataset Requirement

Fill in the predefined functions with your data scraping/parsing code. You may modify/rename each function as you seem fit, but you must provide at least 3 separate functions that clean each of your required datasets.


In [ ]:
import pandas as pd
import numpy as np

def downloaded_dataset_parser(file_path="/Most-Recent-Cohorts-Institution.csv"):
    #load dataset
    scorecard1 = pd.read_csv(file_path, low_memory=False)

    #keep only columns needed for the project
    wanted_cols = [
        "UNITID",
        "INSTNM",
        "STABBR",
        "ZIP",
        "CONTROL",
        "TUITIONFEE_IN",
        "TUITIONFEE_OUT",
        "DEBT_MDN",
        "ADM_RATE",
        "SAT_AVG",
        "C150_4",
        "C150_L4"
    ]

    scorecard = scorecard1[wanted_cols].copy()

    #replace common "missing" strings with NaN
    missing_tokens = ["NULL", "NaN", "nan", "PrivacySuppressed", "Not Available","N/A", "n/a", "", " "]
    scorecard = scorecard.replace(missing_tokens, np.nan)

    #combine columns for graduation rate
    for row in scorecard.index:
        if pd.isna(scorecard.loc[row, "C150_4"]):
          scorecard.loc[row,"C150_4"] = scorecard.loc[row,"C150_L4"]
    scorecard.drop(columns="C150_L4", inplace=True)
    scorecard.rename(columns={"C150_4": "GRAD_RATE"}, inplace=True)

    grad_col = "GRAD_RATE"

    #clean text columns
    if "INSTNM" in scorecard.columns:
        scorecard["INSTNM"] = scorecard["INSTNM"].astype(str).str.strip()

    if "STABBR" in scorecard.columns:
        scorecard["STABBR"] = scorecard["STABBR"].astype(str).str.strip().str.upper()

    if "ZIP" in scorecard.columns:
        scorecard["ZIP"] = scorecard["ZIP"].astype(str).str.extract(r"(\d+)")[0].str.zfill(5)

    #convert numeric columns
    numeric_cols = ["TUITIONFEE_IN", "TUITIONFEE_OUT", "DEBT_MDN", "ADM_RATE", "SAT_AVG"]
    if grad_col is not None:
        numeric_cols.append(grad_col)

    for col in numeric_cols:
        scorecard[col] = pd.to_numeric(scorecard[col], errors="coerce")

    # Remove exact duplicate schools by UNITID
    scorecard = scorecard.drop_duplicates(subset=["UNITID"])

    #fill missing tuition/debt values using state + control medians first, then state medians
    fill_cols = ["TUITIONFEE_IN", "TUITIONFEE_OUT", "DEBT_MDN"]
    for col in fill_cols:
        if "STABBR" in scorecard.columns and "CONTROL" in scorecard.columns:
              scorecard[col] = scorecard[col].fillna(scorecard.groupby(["STABBR", "CONTROL"])[col].transform("median"))

    #fill missing graduation column using state medians
    if grad_col is not None and "STABBR" in scorecard.columns:
        scorecard[grad_col] = scorecard[grad_col].fillna(scorecard.groupby("STABBR")[grad_col].transform("median"))

    #fill missing admission rate / SAT with state medians
    for row in ["ADM_RATE", "SAT_AVG"]:
        scorecard[col] = scorecard[col].fillna(scorecard.groupby("STABBR")[col].transform("median"))

    for row in ["TUITIONFEE_IN", "TUITIONFEE_OUT", "DEBT_MDN", "SAT_AVG"]:
        scorecard[col] = scorecard[col].round(2)

    for row in ["ADM_RATE"]:
        scorecard[col] = scorecard[col].round(4)

    if grad_col is not None and grad_col in scorecard.columns:
        scorecard[grad_col] = scorecard[grad_col].round(4)

    #remove unreasonable negatives
    for col in numeric_cols:
        scorecard.loc[scorecard[col] < 0, col] = np.nan

    #export cleaned dataset
    scorecard.to_csv("cleaned_scorecard.csv", index=False)

    return scorecard


############ Function Call ############
downloaded_dataset_parser()

,UNITID,INSTNM,STABBR,ZIP,CONTROL,TUITIONFEE_IN,TUITIONFEE_OUT,DEBT_MDN,ADM_RATE,SAT_AVG,GRAD_RATE
0,100654,Alabama A & M University,AL,35762,1,10024.0,18634.0,16600.0,0.6622,947.0,0.2874
1,100663,University of Alabama at Birmingham,AL,35294,1,8832.0,21864.0,15832.0,0.8842,1251.0,0.6260
2,100690,Amridge University,AL,36117,2,19688.0,19688.0,13385.0,NaN,NaN,0.4000
3,100706,University of Alabama in Huntsville,AL,35899,1,11770.0,24662.0,13905.0,0.7425,1321.0,0.6191
4,100724,Alabama State University,AL,36104,1,11248.0,19576.0,17500.0,0.9564,977.0,0.3018
...,...,...,...,...,...,...,...,...,...,...,...
6424,49382201,College Unbound - Newport/Aquidneck Island,RI,028400000,2,10488.0,10488.0,7690.0,NaN,NaN,0.7021
6425,49425001,Valley College - Fairlawn - School of Nursing,OH,443333631,3,16699.0,16699.0,9418.0,NaN,NaN,0.5979
6426,49501301,Western Maricopa Education Center - Southwest ...,AZ,85326,1,2358.0,8958.0,8000.0,NaN,NaN,0.5484
6427,49501302,Western Maricopa Education Center - Northeast ...,AZ,85027,1,2358.0,8958.0,8000.0,NaN,NaN,0.5484


## JSON/API Requirement


In [ ]:
import requests
import pandas as pd
import numpy as np

def api_dataset_parser(year=2023, save_path="cleaned_census_income.csv"):
    # this is the census acs 5-year api endpoint
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        # these are the columns i want back from the api
        "get": "NAME,B19013_001E,B19013_001M",
        # this gets all states , queries come from census query components reference
        "for": "state:*"
    }

    # calling the api and turning the response into json
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    # first row has the headers, the rest are the actual rows
    headers = data[0]
    rows = data[1:]
    census_df = pd.DataFrame(rows, columns=headers)

    # renaming columns so they are easier to read later
    census_df = census_df.rename(columns={
        "NAME": "stateName",
        "B19013_001E": "medianHouseholdIncome",
        "B19013_001M": "incomeMarginOfError",
        "state": "stateFips"
    })

    # replacing common missing values with nan
    census_df = census_df.replace(["NULL", "", " "], np.nan)

    # converting numeric columns from strings into actual numbers
    census_df["medianHouseholdIncome"] = pd.to_numeric(census_df["medianHouseholdIncome"], errors="coerce")
    census_df["incomeMarginOfError"] = pd.to_numeric(census_df["incomeMarginOfError"], errors="coerce")

    # making sure state fips codes stay 2 digits
    census_df["stateFips"] = census_df["stateFips"].astype(str).str.zfill(2)

    # this dictionary helps match full state names to abbreviations
    state_name_to_abbr = {
        "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
        "California": "CA", "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE",
        "District of Columbia": "DC", "Florida": "FL", "Georgia": "GA", "Hawaii": "HI",
        "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA", "Kansas": "KS",
        "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
        "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS",
        "Missouri": "MO", "Montana": "MT", "Nebraska": "NE", "Nevada": "NV",
        "New Hampshire": "NH", "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
        "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH", "Oklahoma": "OK",
        "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
        "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT",
        "Vermont": "VT", "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
        "Wisconsin": "WI", "Wyoming": "WY", "Puerto Rico": "PR"
    }

    # making a new column with state abbreviations so it can match scorecard later
    census_df["STABBR"] = census_df["stateName"].map(state_name_to_abbr)

    # keeping only rows where the abbreviation was found
    census_df = census_df[census_df["STABBR"].notna()].copy()

    # rounding numeric columns so they look cleaner
    census_df["medianHouseholdIncome"] = census_df["medianHouseholdIncome"].round(2)
    census_df["incomeMarginOfError"] = census_df["incomeMarginOfError"].round(2)

    # saving the cleaned dataframe to a csv file
    census_df.to_csv(save_path, index=False)

    print("cleaned census income shape:", census_df.shape)
    print(census_df.head())

    return census_df


############ function call ############
api_dataset_parser()

cleaned census income shape: (52, 5)
    stateName  medianHouseholdIncome  incomeMarginOfError stateFips STABBR
0     Alabama                  62027                  400        01     AL
1      Alaska                  89336                 1374        02     AK
2     Arizona                  76872                  414        04     AZ
3    Arkansas                  58773                  503        05     AR
4  California                  96334                  298        06     CA


,stateName,medianHouseholdIncome,incomeMarginOfError,stateFips,STABBR
0,Alabama,62027,400,01,AL
1,Alaska,89336,1374,02,AK
2,Arizona,76872,414,04,AZ
3,Arkansas,58773,503,05,AR
4,California,96334,298,06,CA
5,Colorado,92470,483,08,CO
6,Connecticut,93760,669,09,CT
7,Delaware,82855,1234,10,DE
8,District of Columbia,106287,1803,11,DC
9,Florida,71711,282,12,FL


## HTML Requirement

In [ ]:
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import re
import requests
def html_parser():
  url = "https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_educational_attainment"
  #stuff we got from the homework to access the wikipedia page
  agent = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/97.0.4692.99 Safari/537.36"}
  res = requests.get(url, headers = agent)

  #accessing the specific table we want
  soup = BeautifulSoup(res.text,"html.parser")
  table = soup.find_all("table")[1]

  #getting and fixing the header
  header = table.find_all("th")
  header = [i.text.strip("\n") for i in header]
  header = [i.replace(",","") for i in header]
  header = [i.replace("%","") for i in header]
  header=header[:5]+header[-8:]
  total=header[-8:]
  new_h=[]
  for r in range(5):
    if r in [2,3,4]:
      new_h.append(header[r]+"(Population)")
      new_h.append(header[r]+"(Percent)")
    else:
      new_h.append(header[r])

  #finding all the rows
  row = table.find_all('tr')

  #accessing just the text in each cell in each row and removing "\n" "," and "%"
  for i in range(len(row)):
    row[i] = row[i].find_all("td")
    for j in range(len(row[i])):
      row[i][j] = row[i][j].text.strip("\n")
      row[i][j] = row[i][j].replace(",","")
      row[i][j] = row[i][j].replace("%","")

  #put the info into a dataframe
  df = pd.DataFrame(row,columns=new_h)

  #remove empty rows
  df=df.iloc[2:]
  df.reset_index(drop=True,inplace=True)

  #add the last row back in
  df.iloc[-1] = total

  #make string data into floats and ints and percents into percents
  df.iloc[:,3]=df.iloc[:,3].astype(float)/100
  df.iloc[:,5]=df.iloc[:,5].astype(float)/100
  df.iloc[:,-1]=df.iloc[:,-1].astype(float)/100
  df.iloc[:,1]=df.iloc[:,1].astype(int)
  df.iloc[:,2]=df.iloc[:,2].astype(int)
  df.iloc[:,4]=df.iloc[:,4].astype(int)
  df.iloc[:,6]=df.iloc[:,6].astype(int)

  return df

############ Function Call ############
html_parser()

,State or Territory,Population over the age of 25,With a High School Diploma or higher(Population),With a High School Diploma or higher(Percent),With a Bachelor's Degree or higher(Population),With a Bachelor's Degree or higher(Percent),With an Advanced Degree(Population),With an Advanced Degree(Percent)
0,Alabama,3451208,3034741,0.8793,946766,0.2743,374490,0.1085
1,Alaska,485779,453289,0.9331,159290,0.3279,57868,0.1191
2,Arizona,4980297,4430934,0.8897,1615081,0.3243,629408,0.1264
3,Arkansas,2037763,1806873,0.8867,515034,0.2527,190897,0.0937
4,California,26909869,22724990,0.8445,9737817,0.3619,3779787,0.1405
5,Colorado,4044182,3738208,0.9243,1796240,0.4442,688931,0.1704
6,Connecticut,2534376,2309185,0.9111,1067671,0.4213,479981,0.1894
7,Delaware,711104,649638,0.9136,253292,0.3562,106997,0.1505
8,District of Columbia,478774,444233,0.9279,301852,0.6305,181081,0.3782
9,Florida,15762122,14153579,0.8979,5226846,0.3316,1978625,0.1255


## Additional Dataset Parsing/Cleaning Functions

Write any supplemental (optional) functions here.

In [ ]:
def extra_source1():
    pass


############ Function Call ############
extra_source1()

In [ ]:
# Define further extra source functions as necessary

# Inconsistencies

For each inconsistency (NaN, null, duplicate values, empty strings, etc.) you discover in your datasets, write at least 2 sentences stating the significance, how you identified it, and how you handled it.

1. In the downloaded dataset, some columns such as tuition and median student debt contained values like “PrivacySuppressed” and “NULL” instead of numbers. These values come in as strings and prevent us from performing calculations or converting the column to numeric types. To handle this inconsistency, we replaced these values with NaN and then used pd.to_numeric() with errors='coerce' to ensure all remaining values were properly converted to numeric format.

2. In the API dataset, states were represented using full names (e.g., “Georgia”), while the downloaded dataset used state abbreviations (e.g., “GA”). This inconsistency is significant because it prevents the datasets from being merged directly. To handle this, we created a mapping dictionary to convert state names into their corresponding abbreviations so that both datasets use the same format.

3. In the HTML data, our table had numbers that had commas as separators for the thousands place and "%". These numbers come in the format of a string and we cannot cast them to an int or a float without removing these characters. To handle this inconsistency, we used .replace() to replace these characters with empty strings.

4. (if applicable)

5. (if applicable)
